# 00b — Collect Biomedical Domain (JNLPBA)

Adds a **third target domain** — JNLPBA (molecular-biology abstracts; 5 entity types: protein,
DNA, RNA, cell_line, cell_type) — into the same standardized pipeline as WNUT-17 and SciERC.
JNLPBA is the biomedical benchmark used by Zhang et al. (2024), so the results here sit directly
alongside a dataset already discussed in the literature review.

Source: the IOB2 files from the public `cambridgeltl/MTL-Bioinformatics-2016` repository
(`train.tsv`, `devel.tsv`, `test.tsv`), each token-per-line as `token<TAB>BIO-tag` with blank
lines between sentences. Output is written into the **same schema and location** every downstream
notebook already expects: `data/processed/jnlpba/jnlpba_{train,validation,test}.jsonl` with fields
`id`, `tokens`, `tags`, `source_dataset`, `split`.

> If the URLs below ever 404, any IOB2 copy of JNLPBA works — just point `URLS` at it; the parser
> only needs `token<whitespace>tag` lines with blank-line sentence separators.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json, urllib.request
from pathlib import Path
from collections import Counter

PROCESSED = Path('/content/drive/MyDrive/AAI590/data/processed')
OUT_DIR = PROCESSED / 'jnlpba'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE = 'https://raw.githubusercontent.com/cambridgeltl/MTL-Bioinformatics-2016/master/data/JNLPBA'
URLS = {'train': f'{BASE}/train.tsv', 'validation': f'{BASE}/devel.tsv', 'test': f'{BASE}/test.tsv'}
print('writing to:', OUT_DIR)

Mounted at /content/drive
writing to: /content/drive/MyDrive/AAI590/data/processed/jnlpba


## Step 1 — Download + parse IOB2 into the common JSONL schema

In [2]:
def read_iob2(path):
    rows, toks, tags = [], [], []
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if toks:
                    rows.append((toks, tags)); toks, tags = [], []
                continue
            if line.startswith('-DOCSTART-'):
                continue
            parts = line.split('\t')
            if len(parts) < 2:
                parts = line.split()
            toks.append(parts[0]); tags.append(parts[-1])
    if toks:
        rows.append((toks, tags))
    return rows

def save_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + '\n')

split_rows = {}
for split, url in URLS.items():
    raw_fp = OUT_DIR / f'_raw_{split}.tsv'
    urllib.request.urlretrieve(url, raw_fp)
    sents = read_iob2(raw_fp)
    rows = [{'id': f'{split}-{i}', 'tokens': tk, 'tags': tg,
             'source_dataset': 'jnlpba', 'split': split}
            for i, (tk, tg) in enumerate(sents)]
    save_jsonl(OUT_DIR / f'jnlpba_{split}.jsonl', rows)
    split_rows[split] = rows
    print(f'[OK] {split}: {len(rows)} sentences -> jnlpba_{split}.jsonl')

[OK] train: 16807 sentences -> jnlpba_train.jsonl
[OK] validation: 1739 sentences -> jnlpba_validation.jsonl
[OK] test: 3856 sentences -> jnlpba_test.jsonl


## Step 2 — Summary + sanity check (schema, label set, entity types)

In [3]:
summary = {'dataset': 'jnlpba', 'splits': {}, 'label_counts': {}}
for split, rows in split_rows.items():
    lc = Counter(t for r in rows for t in r['tags'])
    summary['splits'][split] = {'sentences': len(rows), 'tokens': sum(len(r['tokens']) for r in rows)}
    summary['label_counts'][split] = dict(lc.most_common())
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

labels = sorted({t for rows in split_rows.values() for r in rows for t in r['tags']})
etypes = sorted({t.split('-', 1)[1] for t in labels if t != 'O'})
print('splits     :', {k: v['sentences'] for k, v in summary['splits'].items()})
print('label set  :', labels)
print('entity types:', etypes)
print('\nExample:', split_rows['train'][0])
print('\n[OK] wrote jnlpba_{train,validation,test}.jsonl + summary.json to', OUT_DIR)

splits     : {'train': 16807, 'validation': 1739, 'test': 3856}
label set  : ['B-DNA', 'B-RNA', 'B-cell_line', 'B-cell_type', 'B-protein', 'I-DNA', 'I-RNA', 'I-cell_line', 'I-cell_type', 'I-protein', 'O']
entity types: ['DNA', 'RNA', 'cell_line', 'cell_type', 'protein']

Example: {'id': 'train-0', 'tokens': ['IL-2', 'gene', 'expression', 'and', 'NF-kappa', 'B', 'activation', 'through', 'CD28', 'requires', 'reactive', 'oxygen', 'production', 'by', '5-lipoxygenase', '.'], 'tags': ['B-DNA', 'I-DNA', 'O', 'O', 'B-protein', 'I-protein', 'O', 'O', 'B-protein', 'O', 'O', 'O', 'O', 'O', 'B-protein', 'O'], 'source_dataset': 'jnlpba', 'split': 'train'}

[OK] wrote jnlpba_{train,validation,test}.jsonl + summary.json to /content/drive/MyDrive/AAI590/data/processed/jnlpba


**Next steps to fold JNLPBA into the study** (each is a small edit + a resumable re-run):

1. **02 / 03** — add `'jnlpba'` to `TARGET_DATASETS`, re-run to build its few-shot splits and
   tokenized inputs (WNUT/SciERC are already cached, so only JNLPBA is computed).
2. **04** — add JNLPBA to the zero-shot cross-domain evaluation (baseline degradation on the new
   domain).
3. **05 (and optionally 07/08)** — add `'jnlpba'`; the grid is resumable, so only the new
   dataset's runs execute.
4. **09** — add `'jnlpba'`; a `TYPE_NOTES['jnlpba']` glossary entry and (given JNLPBA's large test
   set) a fixed ~800-sentence evaluation cap keep the LLM cost near $1–2.
5. **10** — no change; it will pick up JNLPBA automatically from the results files.